In [1]:
# model
from rich import print as rprint
from dotenv import load_dotenv
load_dotenv(override=True)
from langchain.chat_models import init_chat_model
model = init_chat_model(
    model="deepseek-v4-flash", # 模型名称
    extra_body={
        "thinking": {"type": "disabled"}
    }
)

## Memory

短期记忆: 单轮对话的记忆<br>
长期记忆: 跨会话的记忆

### Short-term memory （State） checkpointer

#### 1、基于内存的短期记忆

In [2]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver  # 内存级的记忆存储 (本质上是message列表拼接)

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(), # 让agent具备记忆能力
)

thread_config = {
    "configurable": {"thread_id": "id_1"} # 同一个thread_id共享一个记忆
}

response1 = agent.invoke(
    {"messages":
        [
            HumanMessage(content="你好我的名字是Reed")
        ]
    },
    config = thread_config
)
print(response1["messages"][-1].content)

# rprint(agent.get_state(thread_config))

response2 = agent.invoke(
    {"messages":
        [
            HumanMessage(content="我叫什么名字")
        ]
    },
    config = thread_config
)
print(response2["messages"][-1].content)

# rprint(agent.get_state(thread_config))

你好，Reed！很高兴认识你。😊

我是DeepSeek，一个由深度求索公司开发的AI助手。我可以帮你解答问题、进行对话、处理文档，或者只是陪你聊聊天。

有什么我可以帮你的吗？无论是学习、工作还是生活上的问题，都欢迎随时问我！
你的名字是**Reed**，是你刚才告诉我的哦！😊 

如果你希望我记住这个名字，在接下来的对话中都可以用“Reed”来称呼你。需要我帮你做些什么吗？


#### 2、基于数据库的短期记忆

In [4]:
import sqlite3
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.sqlite import SqliteSaver

connection = sqlite3.connect("../resources/db/checkpoint.db", check_same_thread=False)
checkpointer = SqliteSaver(connection)
checkpointer.setup() # 建表

agent = create_agent(
    model=model,
    checkpointer=checkpointer, # 让agent具备记忆能力
)

thread_config = {
    "configurable": {"thread_id": "id_2"} # 同一个thread_id共享一个记忆
}

response2 = agent.invoke(
    {"messages":
        [
            HumanMessage(content="你好你的名字是Alice")
        ]
    },
    config = thread_config
)
print(response2["messages"][-1].content)

response2 = agent.invoke(
    {"messages":
        [
            HumanMessage(content="你叫什么名字")
        ]
    },
    config = thread_config
)
print(response2["messages"][-1].content)


你好呀！是的，我的名字是**Alice**，很高兴再次和你打招呼！😊 有什么想聊的或者需要帮忙的吗？
我的名字是**Alice**！😊 很高兴再次告诉你～有什么需要帮忙的吗？


### Long-term memory（Store）